# 02 - Oberflächen-Phasentabellen pro Miller-Index

Dieses Notebook berechnet die Oberflächen-Phasentabellen für die Miller-Indizes
`100`, `110`, `111` und `211`. Ausgangspunkt sind wieder die HDF-Dateien.

Die Oberfläche wird pro Fläche verglichen. Für jede Temperatur und jedes
`pH2O/pH2` wird die korrigierte freie Energie pro Fläche berechnet; die
Struktur mit dem kleinsten Wert definiert das stabile Feld.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

# Compatibility shim for HDF files written in a different NumPy/PyTables stack.
# Keep this cell at the top before calling pd.read_hdf.
try:
    import numpy.core as npc
    sys.modules.setdefault("numpy._core", npc)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
except Exception as exc:
    print("NumPy compatibility shim skipped:", exc)

DATA_ROOT = Path(r"/Users/dk2994/Desktop/Uni/Journal/Thesis/Notebooks/Surface Alloys")
PROJECT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI")
OUTPUT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI/notebooks/phase_diagram_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

HDF_FILES = {
    "bulk": DATA_ROOT / "CuGabulk.hdf",
    "bulk_oxide": DATA_ROOT / "CuGabulk_oxide.hdf",
    "surface_100": DATA_ROOT / "CuGasurf_100.hdf",
    "surface_110": DATA_ROOT / "CuGasurf_110.hdf",
    "surface_111": DATA_ROOT / "CuGasurf_111.hdf",
    "surface_211": DATA_ROOT / "CuGasurf_211.hdf",
}

def read_onepiece_hdf(path, key="df"):
    """Read a OnePiece-exported pandas HDF table.

    OnePiece stores simulation records in a pandas DataFrame. The HDF files in
    this tutorial are read with pd.read_hdf(filename, key="df").
    """
    path = Path(path)
    frame = pd.read_hdf(path, key=key)
    frame.attrs["source_hdf"] = str(path)
    return frame

def formula_counts(formula):
    if not isinstance(formula, str):
        return {}
    counts = {}
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts

In [ ]:
import matplotlib.pyplot as plt

SURFACE_KEYS = ["surface_100", "surface_110", "surface_111", "surface_211"]
surface_frames = {key: read_onepiece_hdf(HDF_FILES[key]) for key in SURFACE_KEYS}
{key: frame.shape for key, frame in surface_frames.items()}

## Chemische Potentiale und Korrekturformel

In [ ]:
CONSTANTS = {
    "H2_E": -6.737,
    "H2O_E": -12.062,
    "H2_S": 1.5044e-3,
    "H2O_S": 2.2198e-3,
    "kb": 8.617333262145e-5,
}

T_values = np.linspace(300, 1100, 161)
log10_ratio_values = np.linspace(-16, 4, 201)
TT, YY_log10 = np.meshgrid(T_values, log10_ratio_values)
LOGR_NATURAL = np.log(10.0) * YY_log10

muO = (
    CONSTANTS["H2O_E"]
    - CONSTANTS["H2_E"]
    - TT * (CONSTANTS["H2O_S"] - CONSTANTS["H2_S"])
    + CONSTANTS["kb"] * TT * LOGR_NATURAL
)
muGa = -1.5 * muO - 11.275508775
muZn = -3.9596642877 - muO

def first_existing(df, names, default=None):
    for name in names:
        if name in df.columns:
            return name
    return default

def corrected_surface_energy_grid(df):
    base_col = first_existing(df, ["form_G", "form_G_per_Area", "E"])
    area_col = first_existing(df, ["Area"], default=None)
    dga_col = first_existing(df, ["delta_Ga"], default=None)
    dzn_col = first_existing(df, ["delta_Zn", "delta_Cu"], default=None)
    muga_col = first_existing(df, ["mu_Ga"], default=None)
    muzn_col = first_existing(df, ["mu_Zn", "mu_Cu"], default=None)

    grids = []
    for _, row in df.iterrows():
        base = float(row[base_col])
        if base_col.endswith("per_Area"):
            area = 1.0
        else:
            area = float(row[area_col]) if area_col else 1.0
        dga = float(row[dga_col]) if dga_col and pd.notna(row[dga_col]) else 0.0
        dzn = float(row[dzn_col]) if dzn_col and pd.notna(row[dzn_col]) else 0.0
        muga_ref = float(row[muga_col]) if muga_col and pd.notna(row[muga_col]) else 0.0
        muzn_ref = float(row[muzn_col]) if muzn_col and pd.notna(row[muzn_col]) else 0.0
        corrected = base + dga * (muga_ref - muGa) + dzn * (muzn_ref - muZn)
        grids.append(corrected / area)
    return np.stack(grids)

## Stabile Phasen pro Miller-Index

In [ ]:
def surface_summary_for(key, df):
    hkl = key.split("_")[-1]
    energy = corrected_surface_energy_grid(df)
    stable_index = np.nanargmin(energy, axis=0)
    stable_energy = np.nanmin(energy, axis=0)
    records = []
    for phase_id in np.unique(stable_index):
        mask = stable_index == phase_id
        row = df.iloc[int(phase_id)]
        short_label = f"{row.get('Monolayer_alloy', np.nan):.1f}% ML" if pd.notna(row.get("Monolayer_alloy", np.nan)) else str(row.get("Name"))
        records.append({
            "hkl": hkl,
            "phase_id": int(phase_id),
            "Name": row.get("Name", f"phase {phase_id}"),
            "Formula": row.get("Formula", ""),
            "phase_label": f"{short_label} · {row.get('Name', phase_id)}",
            "short_label": short_label,
            "Ga": row.get("Ga", np.nan),
            "Cu": row.get("Cu", np.nan),
            "Monolayer_alloy": row.get("Monolayer_alloy", np.nan),
            "stable_grid_fraction": float(mask.mean()),
            "stable_percent": float(100 * mask.mean()),
            "T_min_stable_K": float(TT[mask].min()),
            "T_max_stable_K": float(TT[mask].max()),
            "log10_ratio_min_stable": float(YY_log10[mask].min()),
            "log10_ratio_max_stable": float(YY_log10[mask].max()),
            "min_G_per_Area_eV_A2": float(stable_energy[mask].min()),
        })
    return pd.DataFrame(records).sort_values("stable_grid_fraction", ascending=False), stable_index

surface_summaries = {}
surface_indices = {}
for key, df in surface_frames.items():
    summary, stable_index = surface_summary_for(key, df)
    surface_summaries[key] = summary
    surface_indices[key] = stable_index
    summary.to_csv(OUTPUT_ROOT / f"tutorial_{key}_stable_phases.csv", index=False)

pd.concat(surface_summaries, names=["dataset"]).reset_index(level=0).head(20)

## Ein einzelnes Oberflächenpanel ansehen

In [ ]:
key = "surface_211"
stable_index = surface_indices[key]
summary = surface_summaries[key]
stable_phase_ids = np.unique(stable_index)
remap = {old: new for new, old in enumerate(stable_phase_ids)}
stable_compact = np.vectorize(remap.get)(stable_index)

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
mesh = ax.pcolormesh(TT, YY_log10, stable_compact, cmap=plt.get_cmap("tab20", len(stable_phase_ids)), shading="auto")
ax.set_title("Surface 211 stable phase fields")
ax.set_xlabel("Temperature T [K]")
ax.set_ylabel("log10(pH2O/pH2)")
cbar = fig.colorbar(mesh, ax=ax, ticks=np.arange(len(stable_phase_ids)) + 0.5)
cbar.ax.set_yticklabels([str(surface_frames[key].iloc[i].get("Name", i)) for i in stable_phase_ids])
fig.savefig(OUTPUT_ROOT / "tutorial_surface_211_phase_fields.png", dpi=180)
plt.show()
summary

In [ ]:
surface_all = pd.concat(surface_summaries.values(), ignore_index=True)
surface_all.to_csv(OUTPUT_ROOT / "tutorial_surface_all_stable_phases.csv", index=False)
surface_all